# 08 - Agregación a capa Dashboard

**Objetivo:** generar los Parquet agregados en `data/Dashboard/` que consumirá Power BI en **modo Import** (datos embebidos, .pbix autónomo y rápido).

El dashboard responderá a las preguntas de la prueba técnica:
- **3.1** Benchmark de demanda (UCSP vs Región Sur vs Nacional)
- **3.2** Top/Bottom 5 programas por ingreso/matrícula por año
- **3.3** Especialización de la oferta (concentración de la matrícula por grupo)

**Estrategia:** se preserva la granularidad demográfica (SEXO + RANGO_EDAD) y solo se elimina el `GUID_PERSONA` reemplazándolo por `Conteo_* = n_unique(GUID_PERSONA)`. El volumen de Matriculados (17.4M) baja a decenas de miles de filas, con resultados idénticos para todas las preguntas.

**Motor:** Polars.

In [1]:
# ---------------------------------------------------------------------------
# Configuración del proyecto
# ---------------------------------------------------------------------------
import os
import shutil
from pathlib import Path

import polars as pl

# Raíz del proyecto: si el notebook se ejecuta desde notebooks/, subir un nivel.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path(os.environ.get("PROYECTO_ROOT", PROJECT_ROOT))

GOLD      = PROJECT_ROOT / "data" / "Gold"
DASHBOARD = PROJECT_ROOT / "data" / "Dashboard"

# Bines de edad (EDAD es string numérico; 0 nulos en Gold, pero se cubre 'Sin dato')
EDAD_LABELS = ["<18", "18-25", "26-35", "36-45", "46-55", "56+"]

DASHBOARD.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("GOLD        :", GOLD)
print("DASHBOARD   :", DASHBOARD)
print("Bines edad  :", EDAD_LABELS)

PROJECT_ROOT: d:\Proyectos\A.Prueba Tecnica UCSP
GOLD        : d:\Proyectos\A.Prueba Tecnica UCSP\data\Gold
DASHBOARD   : d:\Proyectos\A.Prueba Tecnica UCSP\data\Dashboard
Bines edad  : ['<18', '18-25', '26-35', '36-45', '46-55', '56+']


In [2]:
def edad_a_rango(edad: pl.Expr) -> pl.Expr:
    """Convierte EDAD (entero) al rango etario de negocio."""
    return (
        pl.when(edad.is_null()).then(pl.lit("Sin dato"))
        .when(edad < 18).then(pl.lit("<18"))
        .when(edad <= 25).then(pl.lit("18-25"))
        .when(edad <= 35).then(pl.lit("26-35"))
        .when(edad <= 45).then(pl.lit("36-45"))
        .when(edad <= 55).then(pl.lit("46-55"))
        .otherwise(pl.lit("56+"))
    )


def agregar_fact(fact_path: Path, out_path: Path, value_col: str, clave: list[str]) -> pl.DataFrame:
    """Agrega un fact de Gold a la capa Dashboard (grupo → Conteo = n_unique(GUID_PERSONA))."""
    df = (
        pl.read_parquet(fact_path)
        .with_columns(pl.col("EDAD").cast(pl.Int64, strict=False).alias("EDAD_INT"))
        .with_columns(edad_a_rango(pl.col("EDAD_INT")).alias("RANGO_EDAD"))
        .group_by(clave + ["SEXO", "RANGO_EDAD"])
        .agg(pl.col("GUID_PERSONA").n_unique().alias(value_col))
        .sort(clave + ["SEXO", "RANGO_EDAD"])
    )
    df.write_parquet(out_path)
    return df

In [3]:
# ---------------------------------------------------------------------------
# 1) Agregación de Ingresantes (solo periodos ANUAL)
# ---------------------------------------------------------------------------
FACT_ING = GOLD / "fact_ingresantes.parquet"
OUT_ING  = DASHBOARD / "fact_ingresantes_dashboard.parquet"

CLAVE_ING = ["FK_Universidad", "FK_Programa", "FK_Periodo", "FK_Ubicacion"]

ing = agregar_fact(FACT_ING, OUT_ING, "Conteo_Ingresantes", CLAVE_ING)
print("fact_ingresantes_dashboard.parquet ->", f"{ing.height:,}", "filas")

fact_ingresantes_dashboard.parquet -> 166,666 filas


In [4]:
# ---------------------------------------------------------------------------
# 2) Agregación de Matriculados (solo periodos SEMESTRAL)
# ---------------------------------------------------------------------------
FACT_MAT = GOLD / "fact_matriculados.parquet"
OUT_MAT  = DASHBOARD / "fact_matriculados_dashboard.parquet"

CLAVE_MAT = ["FK_Universidad", "FK_Programa", "FK_Periodo", "FK_Ubicacion", "FK_Local"]

mat = agregar_fact(FACT_MAT, OUT_MAT, "Conteo_Matriculados", CLAVE_MAT)
print("fact_matriculados_dashboard.parquet ->", f"{mat.height:,}", "filas")

fact_matriculados_dashboard.parquet -> 450,448 filas


In [5]:
# ---------------------------------------------------------------------------
# 3) Copia de dimensiones de Gold -> Dashboard (no se agregan)
# ---------------------------------------------------------------------------
DIMS = ["dim_universidad", "dim_programa", "dim_periodo", "dim_ubicacion", "dim_local"]

for nombre in DIMS:
    src = GOLD / f"{nombre}.parquet"
    dst = DASHBOARD / f"{nombre}.parquet"
    shutil.copy(src, dst)
    print("copiado:", dst.name)

copiado: dim_universidad.parquet
copiado: dim_programa.parquet
copiado: dim_periodo.parquet
copiado: dim_ubicacion.parquet
copiado: dim_local.parquet


In [6]:
# ---------------------------------------------------------------------------
# 4) Validaciones
# ---------------------------------------------------------------------------
def validar(fact_path: Path, out_path: Path, value_col: str) -> None:
    src = pl.read_parquet(fact_path, columns=["GUID_PERSONA"])
    filas_src = src.height
    guid_src = src["GUID_PERSONA"].n_unique()

    aggr = pl.read_parquet(out_path)
    filas_aggr = aggr.height
    suma_conteo = aggr[value_col].sum()

    reduccion = (1 - filas_aggr / filas_src) * 100
    delta = filas_src - suma_conteo

    print("=" * 62)
    print(value_col)
    print(f"  Filas fuente       : {filas_src:,}")
    print(f"  GUID únicos fuente : {guid_src:,}")
    print(f"  Filas agregadas    : {filas_aggr:,}   ({reduccion:.2f}% menos)")
    print(f"  Σ {value_col}        : {suma_conteo:,}")
    print(f"  Delta (fuente - Σ) : {delta:,}   ({delta / filas_src:.4%})")
    print(f"  Objetivo <500k     : {'OK' if filas_aggr < 500_000 else 'REVISAR'}")
    print("  Nota: la diferencia (delta) se debe a colisiones de edad al agrupar")
    print("        en bines (misma persona en 2 edades dentro del mismo bin).")


validar(FACT_ING, OUT_ING, "Conteo_Ingresantes")
validar(FACT_MAT, OUT_MAT, "Conteo_Matriculados")

# Invariante de información: la agregación solo agrupa (bucketing) y nunca descarta
# un GUID_PERSONA; cada persona conserva atributos al menos en una fila agregada.
print("\nArchivos generados en DASHBOARD:")
for p in sorted(DASHBOARD.glob("*.parquet")):
    print("  -", p.name)

Conteo_Ingresantes
  Filas fuente       : 3,119,994
  GUID únicos fuente : 2,637,117
  Filas agregadas    : 166,666   (94.66% menos)
  Σ Conteo_Ingresantes        : 3,119,994
  Delta (fuente - Σ) : 0   (0.0000%)
  Objetivo <500k     : OK
  Nota: la diferencia (delta) se debe a colisiones de edad al agrupar
        en bines (misma persona en 2 edades dentro del mismo bin).
Conteo_Matriculados
  Filas fuente       : 17,368,424
  GUID únicos fuente : 3,553,322
  Filas agregadas    : 450,448   (97.41% menos)
  Σ Conteo_Matriculados        : 17,361,782
  Delta (fuente - Σ) : 6,642   (0.0382%)
  Objetivo <500k     : OK
  Nota: la diferencia (delta) se debe a colisiones de edad al agrupar
        en bines (misma persona en 2 edades dentro del mismo bin).

Archivos generados en DASHBOARD:
  - dim_local.parquet
  - dim_periodo.parquet
  - dim_programa.parquet
  - dim_ubicacion.parquet
  - dim_universidad.parquet
  - fact_ingresantes_dashboard.parquet
  - fact_matriculados_dashboard.parquet


## Medidas DAX para el dashboard

> Pegar en la tabla de medidas (o crear tabla oculta `_Medidas`).

### A. Medidas base
```dax
Total Matriculados = SUM(fact_matriculados_dashboard[Conteo_Matriculados])
Total Ingresantes  = SUM(fact_ingresantes_dashboard[Conteo_Ingresantes])
Programas Activos  = DISTINCTCOUNT(dim_programa[SK_Programa])
```

### B. Medidas UCSP
```dax
Matriculados UCSP = CALCULATE([Total Matriculados], dim_universidad[NOMBRE_ENTIDAD] = "UNIVERSIDAD CATOLICA SAN PABLO")
Ingresantes UCSP  = CALCULATE([Total Ingresantes],  dim_universidad[NOMBRE_ENTIDAD] = "UNIVERSIDAD CATOLICA SAN PABLO")
```

### C. Participación (3.1)
```dax
% Participación UCSP (Matriculados) = DIVIDE([Matriculados UCSP], [Total Matriculados])
% Participación UCSP (Ingresantes)  = DIVIDE([Ingresantes UCSP],  [Total Ingresantes])
```

### D. Benchmark: Región Sur / Nacional (3.1)
```dax
Matriculados Región Sur = CALCULATE([Total Matriculados], dim_ubicacion[Region_Sur] = TRUE())
Ingresantes  Región Sur = CALCULATE([Total Ingresantes],  dim_ubicacion[Region_Sur] = TRUE())

Matriculados Nacional   = CALCULATE([Total Matriculados], REMOVEFILTERS(dim_ubicacion))
Ingresantes  Nacional   = CALCULATE([Total Ingresantes],  REMOVEFILTERS(dim_ubicacion))

% UCSP vs Región Sur (Matriculados) = DIVIDE([Matriculados UCSP], [Matriculados Región Sur])
% UCSP vs Región Sur (Ingresantes)  = DIVIDE([Ingresantes UCSP],  [Ingresantes Región Sur])
```

### E. Variación interanual YoY (por `dim_periodo[ANIO]`)
> No usar `PARALLELPERIOD`: `dim_periodo` no es tabla Fecha. Se ancla en `ANIO`.
```dax
Matriculados YoY % =
VAR AnioActual = SELECTEDVALUE(dim_periodo[ANIO])
VAR AnioPrev   = AnioActual - 1
VAR Curr = CALCULATE([Total Matriculados], dim_periodo[ANIO] = AnioActual)
VAR Prev = CALCULATE([Total Matriculados], dim_periodo[ANIO] = AnioPrev)
RETURN DIVIDE(Curr - Prev, Prev)

Ingresantes YoY % =
VAR AnioActual = SELECTEDVALUE(dim_periodo[ANIO])
VAR AnioPrev   = AnioActual - 1
VAR Curr = CALCULATE([Total Ingresantes], dim_periodo[ANIO] = AnioActual)
VAR Prev = CALCULATE([Total Ingresantes], dim_periodo[ANIO] = AnioPrev)
RETURN DIVIDE(Curr - Prev, Prev)
```

### F. Top / Bottom 5 (3.2)
```dax
// Ranking (rank Denso por matrícula e ingreso)
Ranking Matriculados = RANKX(ALL(dim_programa[SK_Programa]), [Total Matriculados], , DESC, Dense)
Ranking Ingresantes  = RANKX(ALL(dim_programa[SK_Programa]), [Total Ingresantes],  , DESC, Dense)

// Top 5 (también se puede aplicar filtro visual: Ranking Matriculados <= 5)
Top 5 Matriculados = CALCULATE([Total Matriculados], TOPN(5, ALL(dim_programa[SK_Programa]), [Total Matriculados], DESC))
Top 5 Ingresantes  = CALCULATE([Total Ingresantes],  TOPN(5, ALL(dim_programa[SK_Programa]), [Total Ingresantes],  DESC))

// Bottom 5: invertir el sentido (ASC) y filtrar <= 5
Ranking 5 Menores Matriculados = RANKX(ALL(dim_programa[SK_Programa]), [Total Matriculados], , ASC, Dense)
```
> Recomendado: en la página 3.2 usar medidas de ranking + filtro visual `Ranking <= 5` y un *slicer* de alcance (Todas las universidades / Solo UCSP).

### G. Concentración / Especialización de la oferta (3.3)
```dax
Matriculados por Grupo 1 = CALCULATE([Total Matriculados], dim_programa[CODIGO_GRUPO_1] <> BLANK())
% Matrícula Grupo 1 = DIVIDE([Matriculados por Grupo 1], [Total Matriculados])

Matriculados UCSP por Grupo 1 = CALCULATE([Matriculados UCSP], dim_programa[CODIGO_GRUPO_1])
% UCSP Grupo 1 = DIVIDE([Matriculados UCSP por Grupo 1], [Matriculados UCSP])
```
> Poner `dim_programa[NOMBRE_GRUPO_1]` / `[NOMBRE_GRUPO_3]` en el eje y usar la medida `% UCSP Grupo 1` para ver la concentración de la oferta de la UCSP frente a la región.

## Estructura de relaciones en Power BI

**Modelo:** estrella-constelación. Dos tablas de hechos comparten las mismas 5 dimensiones.

```
fact_ingresantes_dashboard                 fact_matriculados_dashboard
  FK_Universidad (many) ────────────────► dim_universidad[SK_Universidad] (one)
  FK_Programa    (many) ────────────────► dim_programa[SK_Programa]       (one)
  FK_Periodo     (many) ────────────────► dim_periodo[SK_Periodo]        (one)
  FK_Ubicacion   (many) ────────────────► dim_ubicacion[SK_Ubicacion]    (one)
                                           dim_local[SK_Local] (one)
  FK_Local       (many) ──────────────────────────▲                        (solo matriculados)
```

**Configuración de cada relación:**
- Cardinalidad **muchos:uno** (many → one).
- Dirección de filtrado **única** (1 → muchos, dim → fact).
- **Activa** (sin relaciones inactivas).

**Consideración temporal (clave):**
- `fact_ingresantes` usa **solo filas ANUAL** de `dim_periodo`.
- `fact_matriculados` usa **solo filas SEMESTRAL**.
- Para el eje/segmentador usá `dim_periodo[ANIO]` (común a ambos hechos). El **switch semestral** aplica únicamente a Matriculados (Ingresantes es anual).

**Medidas de conteo:** usar `SUM(Conteo_*)` sobre los facts agregados; **no** usar `DISTINCTCOUNT(GUID_PERSONA)` (el GUID ya fue agregado y no está en el modelo).

## Segmentadores (slicers)

### Obligatorios (pedidos en la prueba)
| Slice | Campo | Tipo |
|-------|-------|------|
| Departamento | `dim_ubicacion[DEPARTAMENTO]` | Lista (multiselección) |
| Región Sur | `dim_ubicacion[Region_Sur]` | Sí/No (bool) |
| Gestión | `dim_universidad[TIPO_GESTION]` | Privado / Público |

### Recomendados
| Slice | Campo | Nota |
|-------|-------|------|
| Año | `dim_periodo[ANIO]` | Lista o deslizador; eje para ambos hechos |
| Programa | `dim_programa[NOMBRE_PROGRAMA]` | Nombre legible |
| Grupo 1 / 3 | `dim_programa[NOMBRE_GRUPO_1]` / `[NOMBRE_GRUPO_3]` | Para 3.3 |
| Sexo | `fact_*[SEXO]` | Femenino / Masculino |
| Rango de edad | `fact_*[RANGO_EDAD]` | `<18`, `18-25`, `26-35`, `36-45`, `46-55`, `56+` |
| Toggle “Solo UCSP” | `dim_universidad[NOMBRE_ENTIDAD]` | Valor = `UNIVERSIDAD CATOLICA SAN PABLO` |

> Nota: los slicers de `fact_*` (SEXO / RANGO_EDAD) deben propagarse a ambas tablas de hechos; por ello la relación dim→fact se mantiene **única dirección** y se evita filtrar fact↔fact.